In [ ]:
import os
import time
import csv
import pandas as pd
import requests
from bs4 import BeautifulSoup

# 1. Standard headers to properly mimic a desktop browser search request
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 2. Safely locate and load the branch parquet database file
file_path = "data/mb_album_artists.parquet"
csv_output_path = "step_test_result.csv"
pickle_output_path = "data/final_artist_df.pkl"

if not os.path.exists(file_path):
    print(f"❌ Error: File '{file_path}' not found! Ensure the table exists in the correct directory.")
else:
    df = pd.read_parquet(file_path)

    # Dynamic fallback checks for column naming conventions
    artist_column = "artist_name" if "artist_name" in df.columns else "name"
    album_column = "album_name" if "album_name" in df.columns else "album"

    # تعریف ستون‌های فایل خروجی
    fieldnames = [
        "Artist", "Album", "Artist_Listeners", "Artist_Scrobbles", 
        "Album_Listeners", "Album_Scrobbles", "Similar Artists", 
        "Artist_URL", "Album_URL"
    ]

    # هوشمندسازی ساخت فایل: اگر فایل از قبل وجود ندارد، هدر را بساز؛ وگرنه هدر جدید ننویس تا دیتای قبلی خراب نشود
    if not os.path.exists(csv_output_path):
        with open(csv_output_path, mode="w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

    print(f"🎬 Total rows in database: {len(df)}")
    
    # 🎯 اصل کار اینجاست: دیتابیس را اسلایس می‌کنیم تا از ردیف 136,000 به بعد شروع شود
    # برای مثال ردیف 136,000 دیتابیس، ایندکس عددی‌اش می‌شود 136000 (چون از 0 شروع می‌شود)
    df_sliced = df.iloc[31880:]
    
    print(f"🚀 Started scraping FROM ROW 13600. Total rows to process now: {len(df_sliced)}")
    print(f"📝 Data will be saved continuously to '{csv_output_path}' after each row.")
    print("-" * 60)

    total_rows = len(df) # برای اینکه شماره‌گذاری پرینت‌ها درست بماند
    
    try:
        # حالا حلقه را روی دیتای برش‌خورده (df_sliced) اجرا می‌کنیم
        for index, row in df_sliced.iterrows():
            raw_artist = str(row[artist_column]).strip()
            raw_album = str(row[album_column]).strip()

            if not raw_artist or raw_artist.lower() == "none" or raw_artist == "nan":
                print(f" ⚠️ Skipping row index {index+1}: Artist value is empty.")
                continue

            artist_url_slug = raw_artist.replace(" ", "+")
            album_url_slug = raw_album.replace(" ", "+")

            artist_url = f"https://www.last.fm/music/{artist_url_slug}"
            album_url = f"https://www.last.fm/music/{artist_url_slug}/{album_url_slug}"

            # نمایش دقیق لوکیشن ردیف در کل دیتابیس اصلی
            print(f"[{index+1}/{total_rows}] Processing: {raw_artist} - {raw_album}")

            artist_listeners, artist_scrobbles = "N/A", "N/A"
            album_listeners, album_scrobbles = "N/A", "N/A"
            similar_artists_str = "None Found"

            # --- STEP A: CRAWL ARTIST PROFILE ---
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    artist_res = requests.get(artist_url, headers=HEADERS, timeout=10)
                    if artist_res.status_code == 200:
                        artist_soup = BeautifulSoup(artist_res.text, "html.parser")

                        art_container = artist_soup.find("div", class_="header-new-info-desktop")
                        if art_container:
                            metadata_items = art_container.find_all("li", class_="header-metadata-tnew-item")
                            for item in metadata_items:
                                title_el = item.find("h4", class_="header-metadata-tnew-title")
                                abbr_el = item.find("abbr", class_="js-abbreviated-counter")
                                if title_el and abbr_el:
                                    label = title_el.text.strip()
                                    if "Listeners" in label:
                                        artist_listeners = abbr_el.get("title")
                                    elif "Scrobbles" in label:
                                        artist_scrobbles = abbr_el.get("title")

                        similar_artists_list = []
                        similar_headings = artist_soup.find_all("h3", class_="catalogue-overview-similar-artists-item-name")
                        for heading in similar_headings:
                            link_element = heading.find("a")
                            if link_element:
                                similar_artists_list.append(link_element.text.strip())
                        if similar_artists_list:
                            similar_artists_str = ", ".join(similar_artists_list)

                        if artist_listeners != "N/A":
                            break

                    print(f"   ⚠️ Artist stats came back N/A. Retrying ({attempt+1}/{max_retries})...")
                    time.sleep(2)
                except Exception as e:
                    print(f"   ❌ Error crawling artist {raw_artist} (Attempt {attempt+1}): {e}")
                    time.sleep(2)

            time.sleep(1)

            # --- STEP B: CRAWL ALBUM PROFILE ---
            if raw_album and raw_album.lower() != "none" and raw_album != "nan":
                for attempt in range(max_retries):
                    try:
                        album_res = requests.get(album_url, headers=HEADERS, timeout=10)
                        if album_res.status_code == 200:
                            album_soup = BeautifulSoup(album_res.text, "html.parser")
                            abbr_tags = album_soup.find_all("abbr", class_="js-abbreviated-counter")
                            
                            if len(abbr_tags) >= 2:
                                album_listeners = abbr_tags[0].get("title", "N/A")
                                album_scrobbles = abbr_tags[1].get("title", "N/A")
                            elif len(abbr_tags) == 1:
                                album_listeners = abbr_tags[0].get("title", "N/A")

                            if album_listeners != "N/A" and album_scrobbles != "N/A":
                                break
                                
                        print(f"   ⚠️ Album stats came back N/A. Retrying ({attempt+1}/{max_retries})...")
                        time.sleep(2)
                    except Exception as e:
                        print(f"   ❌ Error crawling album {raw_album} (Attempt {attempt+1}): {e}")
                        time.sleep(2)

            # --- ذخیره لحظه‌ای دیتای ردیف فعلی در فایل CSV ---
            row_data = {
                "Artist": raw_artist,
                "Album": raw_album,
                "Artist_Listeners": artist_listeners,
                "Artist_Scrobbles": artist_scrobbles,
                "Album_Listeners": album_listeners,
                "Album_Scrobbles": album_scrobbles,
                "Similar Artists": similar_artists_str,
                "Artist_URL": artist_url,
                "Album_URL": album_url,
            }
            
            with open(csv_output_path, mode="a", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writerow(row_data)

            time.sleep(1.5)

        print("\n✅ Sliced rows processed successfully!")

    except KeyboardInterrupt:
        print("\n🛑 Scraping interrupted by user!")

    finally:
        if os.path.exists(csv_output_path):
            backup_df = pd.read_csv(csv_output_path)
            if len(backup_df) > 0:
                os.makedirs("data", exist_ok=True)
                backup_df.to_pickle(pickle_output_path)
                print(f"📦 Backup secured into Pickle: '{pickle_output_path}'")
        print("🏁 Pipeline process closed safely.")

🎬 Total rows in database: 2241402
🚀 Started scraping FROM ROW 13600. Total rows to process now: 2209522
📝 Data will be saved continuously to 'step_test_result.csv' after each row.
------------------------------------------------------------
[31881/2241402] Processing: Brenda Holloway - The Very Best of Brenda Holloway
[31882/2241402] Processing: Twilight Electric - Rawk Hard
[31883/2241402] Processing: Boogie Down Productions - By All Means Necessary
[31884/2241402] Processing: Blaque - Blaque
   ⚠️ Artist stats came back N/A. Retrying (1/3)...
[31885/2241402] Processing: Big Pun - Yeeeah Baby
   ⚠️ Artist stats came back N/A. Retrying (1/3)...
[31886/2241402] Processing: Benefit - Benefit
[31887/2241402] Processing: Nullsleep - The Gameboy Singles 2002
[31888/2241402] Processing: Yuppster - The Okinawa Campaign 1
   ⚠️ Artist stats came back N/A. Retrying (1/3)...
[31889/2241402] Processing: Magellan - Test of Wills
[31890/2241402] Processing: Magellan - Impending Ascension
[31891/224